In [1]:
import torch 
import tqdm
import numpy as np
import networkx as nx

from agfn.gflownet import GFlowNet
from agfn.gym import MAG, LogReward
from agfn.policies import ForwardPolicy, BackwardPolicy


def test_gflownet():
    num_nodes = 5
    # Create a MAG with 5 nodes
    mag = MAG(num_nodes, batch_size=1, log_reward=None)
    pf = ForwardPolicy(num_nodes=num_nodes, hidden_dim=128, num_layers=2)
    pb = BackwardPolicy()
    gfn = GFlowNet(pf, pb)
    gfn.sample(mag)
    return mag.adjacency_matrix

assert isinstance(test_gflownet(), torch.Tensor) 

def test_gflownet_set_posterior():
    num_nodes = 5
    # Create a MAG with 5 nodes
    mag = MAG(num_nodes, batch_size=1, log_reward=None)
    pf = ForwardPolicy(num_nodes=num_nodes, hidden_dim=128, num_layers=2)
    pb = BackwardPolicy()
    gfn = GFlowNet(pf, pb)

    num_edges = (num_nodes * (num_nodes - 1)) // 2
    posterior = torch.ones((num_edges, 4)) / 4
    gfn.set_posterior(posterior)
    gfn.sample(mag)
    return mag.adjacency_matrix

assert isinstance(test_gflownet_set_posterior(), torch.Tensor) 

In [2]:
num_nodes = 5
# Create a simple LogReward
true_mag, samples = MAG.generate_dag_with_bounded_degree(num_nodes, max_degree=2, num_samples=100)

In [3]:
log_reward = LogReward(samples)

## Train a GFlowNet on `log_reward`

In [4]:
pf = ForwardPolicy(num_nodes, hidden_dim=128, num_layers=2)
pb = BackwardPolicy()
gfn = GFlowNet(pf, pb)

log_reward.warmup(gfn, MAG(num_nodes, batch_size=128, log_reward=log_reward), num_batches=100)

In [5]:
opt = torch.optim.Adam([
    {"params": gfn.pf.parameters(), "lr": 1e-3},
    {"params": gfn.pb.parameters(), "lr": 1e-3},
    {"params": gfn.log_z, "lr": 1e-1},
], lr=1e-3)

pbar = tqdm.tqdm(range(int(5e1)), disable=False)
for epoch in pbar:
    initial_state = MAG(num_nodes, batch_size=128, log_reward=log_reward)
    opt.zero_grad()
    loss = gfn(initial_state)
    loss.backward()
    opt.step()
    pbar.set_postfix(loss=loss.item())

100%|██████████| 50/50 [00:01<00:00, 25.96it/s, loss=5.8] 


## Human in the loop

In [6]:
from agfn.hitl import find_next_query, posterior_update, EdgeType

In [7]:
def oracle(query):
    i, j = np.unravel_index(query, (num_nodes, num_nodes))
    amag = nx.adjacency_matrix(true_mag)
    
    if amag[i, j] == 1 and amag[j, i] == 1:
        return EdgeType.LATENT_CONFOUNDING
    elif amag[i, j] == 1:
        return EdgeType.LEFT_CAUSES_RIGHT
    elif amag[j, i] == 1:
        return EdgeType.RIGHT_CAUSES_LEFT
    else:
        return EdgeType.NO_RELATIONSHIP

In [8]:
queries = list()
responses = list()

for _ in range(10):
    query = find_next_query(
        gfn, 
        log_reward, 
        queries=queries, 
        responses=responses, 
        pi=0.95,
        num_iterations=2
    )
    response = oracle(query)
    queries.append(query)
    responses.append(response.value)
    print(f"Query: {query}, Response: {response}")

Query: 1, Response: EdgeType.NO_RELATIONSHIP
Query: 4, Response: EdgeType.NO_RELATIONSHIP
Query: 9, Response: EdgeType.NO_RELATIONSHIP
Query: 3, Response: EdgeType.NO_RELATIONSHIP
Query: 7, Response: EdgeType.NO_RELATIONSHIP
Query: 6, Response: EdgeType.NO_RELATIONSHIP
Query: 2, Response: EdgeType.NO_RELATIONSHIP
Query: 8, Response: EdgeType.NO_RELATIONSHIP
Query: 5, Response: EdgeType.NO_RELATIONSHIP
Query: 0, Response: EdgeType.NO_RELATIONSHIP


In [10]:
# Sample from the resulting posterior
num_edges = (num_nodes * (num_nodes - 1)) // 2
posterior_dist = posterior_update(responses, queries, num_edges, pi=0.95)

gfn.set_posterior(posterior_dist)

In [22]:
initial_state = MAG(num_nodes, batch_size=128, log_reward=log_reward)
mag = gfn.sample(initial_state)
mag.adjacency_matrix

tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 1., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        ...,

        [[0., 0., 0., 1., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 1., 0., 0.],
         [0., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0.]]])